# Compare the Production and destruction profiles for different turbulence models

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from os import makedirs
from glob import glob
from pandas import read_csv
from os.path import join, exists

In [ ]:
# validation @ Ma = 0.73
u_inf = 242.16629

# chord length
chord = 1

# use latex fonts
plt.style.use("default")
plt.rcParams.update({"text.usetex": True, "figure.dpi": 360})

# use these line styles
ls = ["-", "--", "-.", ":"]

In [ ]:
# SALSA vs. exp. data
load_dir = join("/media", "janis", "Elements", "Janis", "2D_buffet_simulation", "URANS_2D_Ma0.73_Re3e6")
save_dir = join("..", "run", "plots", "URANS_validation", "URANS_blockMesh", "SALSA", "revised_new_mesh", "comparison_production_destruction_terms")
cases = ["URANS_SA_alpha3.5deg_blockMesh_newMesh", "URANS_SA_cc_alpha3.5deg_blockMesh_newMesh", "URANS_SALSA_alpha3.5deg_blockMesh_useRmod_useSmod_newMesh"]

legend = [r"$\mathrm{SA}$", r"$\mathrm{SA-CC}$", r"$\mathrm{SALSA}$"]

# use boundary layer thickness for scaling y, else y+
useDelta = True

In [ ]:
# load the line samples for the specified locations and times
locations = ["0.28", "0.45", "0.6", "0.75"]
loc = ["028", "045", "06", "075"]

# timings for mean / min / max cl for the unsteady cases
# SALSA: ["0.952", "0.916", "0.932"] (max, min, mean)
# SA-CC: ["0.935", "0.902", "0.918"] (max, min, mean)
write_time = "0.916"

# map the labels
if write_time == "0.932":
    label = "mean"
    wt_SA_cc = "0.918"
elif write_time == "0.952":
    label = "max"
    wt_SA_cc = "0.934"
else:
    label = "min"
    wt_SA_cc = "0.902"

In [ ]:
# create plot directory
if not exists(save_dir):
    makedirs(save_dir)

In [ ]:
# SA model constants
Cb1 = 0.1355
Cb2 = 0         # for SALSA
Cw2 = 0.3
Cw3 = 2
kappa = 0.41
sigma = 2/3
Cv1 = 7.1
Cv1_3 = Cv1**3

# common free stream quantities
rho0 = 0.957837
mu = 7.7319e-05
gamma = 1.4

# SA-CC
C5 = 3.5

def compute_cc(_rho, _nuTilda, _gamma, line, counter = 0):
    gradU2 = (line["gradU_xx"]**2 + line["gradU_xz"]**2 + line["gradU_zx"]**2 + line["gradU_zz"]**2)
    a2 = _gamma * line["p"].values / _rho

    # debug
    bl, _ = compute_bl_thickness(np.sqrt(line["Ux"].values**2 + line["Uz"].values**2), (line["z"].values - line["z"][0]))

    fix, ax = plt.subplots(figsize=(6, 3))
    ax.plot(a2, (line["z"].values - line["z"][0]) / bl, ls="-", label="$a^2$")
    ax.plot(gradU2, (line["z"].values - line["z"][0]) / bl, ls="-", label=r"$(\nabla U)^2$")
    ax.plot(C5 * _rho * _nuTilda**2, (line["z"].values - line["z"][0]) / bl, ls="-", label=r"$C_5 \rho \tilde{\nu}^2$")
    ax.plot(gradU2/ a2, (line["z"].values - line["z"][0]) / bl, color="black", label=r"$ (\nabla U)^2 / a^2$")
    ax.set_ylabel(r"$y / \delta$")
    ax.set_yscale("log")
    ax.set_xscale("log")
    ax.set_ylim(0, 1)
    ax.legend()
    plt.savefig(join(save_dir, f"CC_vs_line{counter}_{label}.png"))
    plt.close("all")

    return C5 * _rho * _nuTilda**2 * gradU2 / a2

def compute_Sstar(line):
    Sxx = line["gradU_xx"]
    Sxz = 0.5*(line["gradU_xz"] + line["gradU_zx"])
    Szz = line["gradU_zz"]
    trace = Sxx + Szz

    Sdev_xx = Sxx - trace/3
    Sdev_yy = -trace/3
    Sdev_zz = Szz - trace/3
    Sdev_xz = Sxz

    # S^*
    return np.sqrt(2.0 * (Sdev_xx**2 + Sdev_yy**2 + Sdev_zz**2 + 2.0*Sdev_xz**2))

def compute_omega(line):
    dUxdz = line["gradU_xz"].values
    dUzdx = line["gradU_zx"].values

    return np.abs(dUxdz - dUzdx)

def compute_Stilde_SALSA(_rho, _nuTilda, S):
    chi = _nuTilda * _rho / mu
    fv1 = chi**3 / (chi**3 + Cv1_3)
    factor = 1.0 / np.maximum(chi, 1e-15) + fv1

    # Stilde = S^* (or Stilde) * (1/chi + fv1)
    return S * factor

def compute_cw1(_sqrtGamma, cb2):
    return (Cb1 * _sqrtGamma) / kappa**2 + (1 + cb2) / sigma

def compute_Edwards_mod(_rho, _nuTilda, _d, _Stilde):
    psi_ = np.sqrt(rho0 / _rho) * (_nuTilda / (kappa**2 * np.maximum(_d**2, 1e-15)))
    return 1.6 * np.tanh( 0.7 * (psi_ / _Stilde))

def compute_r(_nuTilda, _d, _Stilde):
        return np.minimum(_nuTilda / (kappa**2 * np.maximum(_d**2, 1e-15)) * (1 / _Stilde), 10.0)

def compute_fw(_rho, _nuTilda, _d, _Stilde, edwards: bool):
    if edwards:
        r_ = compute_Edwards_mod(_rho, _nuTilda, _d, _Stilde)
    else:
        r_ = compute_r(_nuTilda, _d, _Stilde)

    g_ = r_ * (1 + Cw2 * (r_**5 - 1))
    return g_ * ( (1 + Cw3**6) / (g_**6 + Cw3**6))**(1/6)

def compute_Stilde_SA(_rho, _nuTilda, _omega, _d):
    chi = _rho * _nuTilda / mu
    fv1 = chi**3 / (chi**3 + Cv1_3)
    fv2 = 1.0 - chi / (1.0 + chi * fv1)

    return _omega + fv2 * _nuTilda/(kappa**2*np.maximum(_d**2, 1e-15))

def compute_bl_thickness(_u, _z):
    # get idx of the maximum and crop everything after the max.
    max_idx = _u.argmax()
    u99 = 0.99 * _u[max_idx]

    # interpolate the z-coordinate at 0.99 * Ue
    _delta = np.interp(u99, _u[:max_idx], _z[:max_idx])
    return _delta, u99

def compute_y_plus(line):
    y = line["z"].values - line["z"].values[0]
    rho_w = line["rho"].values[0]
    tau_w = np.sqrt(line["tau_wx"].values[0]**2 + line["tau_wz"].values[0]**2)
    u_tau = np.sqrt(tau_w / rho_w)
    nu = mu / rho_w

    return y * u_tau / nu, u_tau

def compute_shape_factor(line):
    U = np.sqrt(line["Ux"].values**2 + line["Uz"].values**2)
    rho = line["rho"].values
    y = line["z"].values - line["z"].values[0]

    # Edge velocity: maximum velocity in the profile
    delta, u99 = compute_bl_thickness(U, y)
    edge_idx = np.searchsorted(y, delta)

    # Only integrate up to the edge
    U = U[:edge_idx + 1]
    rho = rho[:edge_idx + 1]
    y = y[:edge_idx + 1]
    Ue = u99
    rho_e = rho[-1]

    # Compressible, density-weighted profiles
    rhoU_ratio = rho * U / (rho_e * Ue)

    # displacement and momentum thickness as well as shape factor
    delta_star = np.trapezoid(1.0 - rhoU_ratio, y)
    theta = np.trapezoid(rhoU_ratio * (1.0 - U / Ue), y)

    H = delta_star / theta

    return H, delta_star, theta

In [ ]:
# initialize empty lists for results
production, destruction, compressibility, sqrtGamma = [], [], [], []
all_z, all_u, all_Stilde, all_ux, all_uz, all_yPlus, all_nut, all_nu, all_nuTilda = [], [], [], [], [], [], [], [], []

for c in range(len(cases)):
    # define the columns to load
    if "SALSA" in cases[c]:
        cols = [0, 2, 3, 4, 7, 8, 9, 11, 12, 14, 15, 17, 18, 20, 23, 24, 26, 30, 32]
        names = ["z", "nuTilda", "nut", "p", "rho", "sqrtGamma", "Ux", "Uz", "Ux_mean", "Uz_mean",
                 "tau_wx", "tau_wz", "U_xx", "U_xz", "U_zz", "gradU_xx", "gradU_xz", "gradU_zx", "gradU_zz"]
    else:
        cols = [0, 2, 3, 4, 7, 8, 10, 11, 13, 14, 16, 17, 19, 22, 23, 25, 29, 31]
        names = ["z", "nuTilda", "nut", "p", "rho", "Ux", "Uz", "Ux_mean", "Uz_mean", "tau_wx", "tau_wz",
                 "U_xx", "U_xz", "U_zz", "gradU_xx", "gradU_xz", "gradU_zx", "gradU_zz"]

    # make sure we compare the same states wrt shock position
    wt = wt_SA_cc if c == 1 else write_time

    # load the line samples
    try:
        files = [glob(join(load_dir, cases[c], "postProcessing", "sample_lines", wt, f"xc_{l}_*.csv"))[0] for l in loc]
    except IndexError:
        # for SA we only have the last dt, since it reaches a steady state anyway
        files = [glob(join(load_dir, cases[c], "postProcessing", "sample_lines", "1.0001", f"xc_{l}_*.csv"))[0] for l in loc]

    lines = [read_csv(f, names=names, header=None, sep=",", skiprows=1, usecols=cols) for f in files]
    z, g, cc, p, des, u, stilde, ux, uz, yplus, nut, nu, nuTilda_tmp = [], [], [], [], [], [], [], [], [], [], [], [], []

    # loop over each sampling line and compute the production & destruction terms
    for s in range(len(locations)):
        # extract common variables
        rho = lines[s]["rho"].values
        nuTilda = lines[s]["nuTilda"].values
        nut.append(lines[s]["nut"].values)
        nu.append(mu / rho)

        # for wall distance omit the zero term, not sure how accurate this is, but shouldn't be much of an issue
        d = (lines[s]["z"].values - lines[s]["z"][0])

        # compute the compressibility correction for SA-CC
        if "SA_cc" in cases[c]:
            cc.append(compute_cc(rho, nuTilda, gamma, lines[s], counter=s))
        else:
            cc.append(np.zeros(len(nuTilda)))

        # compute Sstar or Stilde
        if "SALSA" in cases[c]:
            Sstar = compute_Sstar(lines[s])

            # if SA -> Stilde instead of SA, rest remains the same
            Stilde = compute_Stilde_SALSA(rho, nuTilda, Sstar)

            # compute the production
            g.append(lines[s]["sqrtGamma"])
            p.append(Cb1 * Stilde * nuTilda * rho * lines[s]["sqrtGamma"])
        else:
            omega = compute_omega(lines[s])
            Stilde = compute_Stilde_SA(rho, nuTilda, omega, d)
            p.append(Cb1 * Stilde * nuTilda * rho)
            g.append([])

        # compute destruction
        if "SALSA" in cases[c]:
            cw1 = compute_cw1(lines[s]["sqrtGamma"], Cb2)
            # compute fw
            fw = compute_fw(rho, nuTilda, d, Stilde, True)
        else:
            cw1 = compute_cw1(np.ones(Stilde.shape), 0.622)
            # compute fw
            fw = compute_fw(rho, nuTilda, d, Stilde, False)

        # print shape factor etc.
        H, delta_star, theta = compute_shape_factor(lines[s])
        print(f"x/c = {locations[s]} | "
            f"H = {H:.3f} | "
            f"delta* = {delta_star:.6e} | "
            f"theta = {theta:.6e}")

        z.append(d)
        nuTilda_tmp.append(nuTilda)
        yp, _ = compute_y_plus(lines[s])
        yplus.append(yp)
        stilde.append(Stilde)
        des.append((cw1 * fw) * nuTilda**2 / np.maximum(d**2, 1e-15) * rho)
        u.append(np.sqrt(lines[s]["Ux"].values**2 + lines[s]["Uz"].values**2))
        ux.append(lines[s]["Ux"].values)
        uz.append(lines[s]["Uz"].values)

    all_z.append(z)
    all_nut.append(nut)
    all_nuTilda.append(nuTilda_tmp)
    all_nu.append(nu)
    all_yPlus.append(yplus)
    all_Stilde.append(stilde)
    destruction.append(des)
    production.append(p)
    sqrtGamma.append(g)
    compressibility.append(cc)
    all_u.append(u)
    all_ux.append(ux)
    all_uz.append(uz)

In [ ]:
# plot the corresponding Stilde
fig, ax = plt.subplots(nrows= 1, ncols=4, figsize=(6, 3), sharey="row")

for c in range(len(cases)):
    for i in range(len(locations)):
        # compute approx. BL height
        if useDelta:
            sf, _ = compute_bl_thickness(all_u[c][i], all_z[c][i])
        else:
            sf = all_yPlus[c][i]

        if i == 0:
            ax[i].plot(all_Stilde[c][i], all_z[c][i] / sf if useDelta else sf, zorder=10, color="black", marker="none", ls=ls[c], label=legend[c])
        else:
            ax[i].plot(all_Stilde[c][i], all_z[c][i] / sf if useDelta else sf, zorder=10, color="black", marker="none", ls=ls[c])

        ax[i].grid(visible=True, which="major", linestyle="-", alpha=0.35, color="black", axis="both")
        ax[i].minorticks_on()
        ax[i].tick_params(axis="both", which="minor", bottom=True)
        ax[i].grid(visible=True, which="minor", linestyle="--", alpha=0.25, color="black", axis="both")
        ax[i].set_title(fr"$ x / c = {float(locations[i]):.2f}$")
        # ax[i].set_ylim(0, 0.06)
        # ax[i].set_ylim(0, 1)
        ax[i].set_ylim(0, 25)
        #ax[i].set_xscale("log")
        ax[i].set_xlim(-1e5, 1e6)
        # ax[i].set_yscale("log")

# ax[0].set_ylabel(r"$y / c$")
if useDelta:
    ax[0].set_ylabel(r"$y / \delta$")
else:
    ax[0].set_ylabel(r"$y^+$")

fig.legend(ncol=4, loc="lower center")
fig.tight_layout()
fig.subplots_adjust(bottom=0.25)
plt.savefig(join(save_dir, f"comparison_S_tilde_{label}_scaled_with_{"delta" if useDelta else "yPlus"}.png"))
plt.show()

In [ ]:
# plot the velocity profiles
fig, ax = plt.subplots(nrows= 1, ncols=4, figsize=(6, 3), sharey="row")

for c in range(len(cases)):
    for i in range(len(locations)):
        # compute approx. BL height
        if useDelta:
            sf, ue = compute_bl_thickness(all_u[c][i], all_z[c][i])
        else:
            sf = all_yPlus[c][i]
            ue = u_inf

        if i == 0:
            ax[i].plot(all_ux[c][i] / ue, all_z[c][i] / sf if useDelta else sf, zorder=10, color="black", marker="none", ls=ls[c], label=legend[c])
        else:
            ax[i].plot(all_ux[c][i] / ue, all_z[c][i] / sf if useDelta else sf, zorder=10, color="black", marker="none", ls=ls[c])

        ax[i].grid(visible=True, which="major", linestyle="-", alpha=0.35, color="black", axis="both")
        ax[i].minorticks_on()
        ax[i].tick_params(axis="both", which="minor", bottom=True)
        ax[i].grid(visible=True, which="minor", linestyle="--", alpha=0.25, color="black", axis="both")
        ax[i].set_title(fr"$ x / c = {float(locations[i]):.2f}$")
        # ax[i].set_ylim(0, 1)
        ax[i].set_ylim(1e-4, 1.5)
        # ax[i].set_ylim(0, 25)
        # ax[i].set_xscale("log")
        # ax[i].set_xlim(0, 1e5)
        ax[i].set_yscale("log")

# ax[0].set_ylabel(r"$y / c$")
if useDelta:
    ax[0].set_ylabel(r"$y / \delta$")
else:
    ax[0].set_ylabel(r"$y^+$")
fig.supxlabel(r"$u / U_e$")

fig.legend(ncol=4, loc="upper center")
fig.tight_layout()
fig.subplots_adjust(top=0.8)
plt.savefig(join(save_dir, f"comparison_velocity_profiles_log_{label}_scaled_with_{"delta" if useDelta else "yPlus"}.png"))
plt.show()

In [ ]:
# plot the CC and sqrt gamma profiles to check if the effect is the same
fig, ax = plt.subplots(nrows= 1, ncols=4, figsize=(6, 3), sharey="row")
ax2 = [a.twiny() for a in ax]

for i in range(len(locations)):
    # compute approx. BL height
    if useDelta:
        sf_SAcc, _ = compute_bl_thickness(all_u[1][i], all_z[1][i])
        sf_SALSA, _ = compute_bl_thickness(all_u[2][i], all_z[2][i])
    else:
        sf_SAcc = all_yPlus[1][i]
        sf_SALSA = all_yPlus[2][i]

    if i == 0:
        ax2[i].plot(-compressibility[1][i], all_z[1][i] / sf_SAcc if useDelta else sf_SAcc, zorder=10, color="red", marker="none", ls="--",
                    label=r"$-c_5 \frac{\tilde{\nu}^2}{a^2}\frac{\partial u_i}{\partial x_j}\frac{\partial u_i}{\partial x_j}$")
        ax[i].plot(sqrtGamma[2][i], all_z[2][i] / sf_SALSA  if useDelta else sf_SALSA, zorder=10, color="black", marker="none", ls="-", label=r"$\sqrt\Gamma$")
    else:
        ax2[i].plot(-compressibility[1][i], all_z[1][i] / sf_SAcc if useDelta else sf_SAcc, zorder=10, color="red", marker="none", ls="--")
        ax[i].plot(sqrtGamma[2][i], all_z[2][i] / sf_SALSA if useDelta else sf_SALSA, zorder=10, color="black", marker="none", ls="-")

    ax[i].grid(visible=True, which="major", linestyle="-", alpha=0.35, color="black", axis="both")
    ax[i].minorticks_on()
    ax[i].tick_params(axis="both", which="minor", bottom=True)
    ax[i].grid(visible=True, which="minor", linestyle="--", alpha=0.25, color="black", axis="both")
    ax[i].set_title(fr"$ x / c = {float(locations[i]):.2f}$")
    # ax[i].set_ylim(0, 0.06)
    ax[i].set_ylim(1e-4, 1.5)
    # ax[i].set_ylim(0, 2500)
    ax[i].set_xlim(0.86, 1.14)
    # ax[i].set_xscale("log")
    ax[i].set_yscale("log")

    # second x-axis
    ax2[i].tick_params(axis="x", colors="red")
    ax2[i].set_xlim(-16, 0.1)
    ax2[i].minorticks_on()
    ax2[i].spines["top"].set_color("red")
    ax2[i].tick_params(axis="x", which="minor", colors="red")

# ax[0].set_ylabel(r"$y / c$")
if useDelta:
    ax[0].set_ylabel(r"$y / \delta$")
else:
    ax[0].set_ylabel(r"$y^+$")

fig.legend(ncol=4, loc="lower center")
fig.tight_layout()
fig.subplots_adjust(bottom=0.24)
plt.savefig(join(save_dir, f"sqrtGamma_CC_profiles_with_ratio_log_cl_{label}_scaled_with_{"delta" if useDelta else "yPlus"}.png"))
plt.show()

In [ ]:
# plot the production & destruction terms
fig, ax = plt.subplots(nrows= 1, ncols=4, figsize=(6, 3), sharey="row")

for c in range(len(cases)):
    for i in range(len(locations)):
        # compute approx. BL height
        if useDelta:
            sf, _ = compute_bl_thickness(all_u[c][i], all_z[c][i])
        else:
            sf = all_yPlus[c][i]

        # compute total destruction
        d = destruction[c][i] #+ compressibility[c][i]

        if i == 0:
            ax[i].plot((production[c][i] - d) / (production[c][i] + d), all_z[c][i] / sf  if useDelta else sf, zorder=10, color="black",
                        marker="none", ls=ls[c], label=legend[c])
        else:
            ax[i].plot((production[c][i] - d) / (production[c][i] + d), all_z[c][i] / sf  if useDelta else sf, zorder=10, color="black",
                        marker="none", ls=ls[c])

        ax[i].grid(visible=True, which="major", linestyle="-", alpha=0.35, color="black", axis="both")
        ax[i].minorticks_on()
        ax[i].tick_params(axis="both", which="minor", bottom=True)
        ax[i].grid(visible=True, which="minor", linestyle="--", alpha=0.25, color="black", axis="both")
        ax[i].set_title(fr"$ x / c = {float(locations[i]):.2f}$")
        # ax[i].set_ylim(0, 0.06)
        ax[i].set_ylim(0, 1)
        # ax[i].set_ylim(0, 2500)
        ax[i].set_xlim(-1.8, 1.4)
        # ax[i].set_xscale("log")
        ax[i].set_yscale("log")

# ax[0].set_ylabel(r"$y / c$")
if useDelta:
    ax[0].set_ylabel(r"$y / \delta$")
else:
    ax[0].set_ylabel(r"$y^+$")
fig.legend([r"$(P_{\tilde{\nu}} - D_{\tilde{\nu}}) + (P_{\tilde{\nu}} + D_{\tilde{\nu}})$"], loc="upper center")

fig.legend(ncol=4, loc="lower center")
fig.tight_layout()
fig.subplots_adjust(bottom=0.2, top=0.82)
plt.savefig(join(save_dir, f"production_destruction_ratios_profiles_with_ratio_log_cl_{label}_scaled_with_{"delta" if useDelta else "yPlus"}.png"))
plt.show()

In [ ]:
# plot nut / nu
fig, ax = plt.subplots(nrows= 1, ncols=4, figsize=(6, 3), sharey="row")

for c in range(len(cases)):
    for i in range(len(locations)):
        # compute approx. BL height
        if useDelta:
            sf, _ = compute_bl_thickness(all_u[c][i], all_z[c][i])
        else:
            sf = all_yPlus[c][i]

        if i == 0:
            ax[i].plot(all_nut[c][i] / all_nu[c][i], all_z[c][i] / sf  if useDelta else sf, zorder=10, color="black", marker="none", ls=ls[c], label=legend[c])
            # ax[i].plot(all_nuTilda[c][i] / all_nu[c][i], all_z[c][i] / sf  if useDelta else sf, zorder=10, color="black", marker="none", ls=ls[c], label=legend[c])
        else:
            ax[i].plot(all_nut[c][i] / all_nu[c][i], all_z[c][i] / sf  if useDelta else sf, zorder=10, color="black", marker="none", ls=ls[c])
            # ax[i].plot(all_nuTilda[c][i] / all_nu[c][i], all_z[c][i] / sf  if useDelta else sf, zorder=10, color="black", marker="none", ls=ls[c])

        ax[i].grid(visible=True, which="major", linestyle="-", alpha=0.35, color="black", axis="both")
        ax[i].minorticks_on()
        ax[i].tick_params(axis="both", which="minor", bottom=True)
        ax[i].grid(visible=True, which="minor", linestyle="--", alpha=0.25, color="black", axis="both")
        ax[i].set_title(fr"$ x / c = {float(locations[i]):.2f}$")
        # ax[i].set_ylim(0, 0.06)
        ax[i].set_ylim(0, 1.5)
        # ax[i].set_ylim(0, 2500)
        # ax[i].set_xlim(0, 1)
        # ax[i].set_xscale("log")
        ax[i].set_yscale("log")

# ax[0].set_ylabel(r"$y / c$")
if useDelta:
    ax[0].set_ylabel(r"$y / \delta$")
else:
    ax[0].set_ylabel(r"$y^+$")
fig.legend([r"$\nu_t / \nu$"], loc="upper center")
# fig.legend([r"$\tilde{\nu} / \nu$"], loc="upper center")

fig.legend(ncol=4, loc="lower center")
fig.tight_layout()
fig.subplots_adjust(bottom=0.2, top=0.8)
plt.savefig(join(save_dir, f"nut_vs_nu_ratio_log_cl_{label}_scaled_with_{"delta" if useDelta else "yPlus"}.png"))
# plt.savefig(join(save_dir, f"nuTilda_vs_nu_ratio_log_cl_{label}_scaled_with_{"delta" if useDelta else "yPlus"}.png"))
plt.show()

In [ ]:
# plot the production & destruction terms
fig, ax = plt.subplots(nrows= 1, ncols=4, figsize=(6, 3), sharey="row")

for c in range(len(cases)):
    for i in range(len(locations)):
        # compute approx. BL height
        if useDelta:
            sf, _ = compute_bl_thickness(all_u[c][i], all_z[c][i])
        else:
            sf = all_yPlus[c][i]

        # compute total destruction
        d = destruction[c][i] # + compressibility[c][i]

        if i == 0:
            ax[i].plot(production[c][i] / d, all_z[c][i] / sf  if useDelta else sf, zorder=10, color="black",
                        marker="none", ls=ls[c], label=legend[c])
        else:
            ax[i].plot(production[c][i] / d, all_z[c][i] / sf  if useDelta else sf, zorder=10, color="black",
                        marker="none", ls=ls[c])

        ax[i].grid(visible=True, which="major", linestyle="-", alpha=0.35, color="black", axis="both")
        ax[i].minorticks_on()
        ax[i].tick_params(axis="both", which="minor", bottom=True)
        ax[i].grid(visible=True, which="minor", linestyle="--", alpha=0.25, color="black", axis="both")
        ax[i].set_title(fr"$ x / c = {float(locations[i]):.2f}$")
        # ax[i].set_ylim(0, 0.06)
        ax[i].set_ylim(0, 1)
        # ax[i].set_ylim(0, 2500)
        ax[i].set_xlim(-0.1, 2)
        # ax[i].set_xscale("log")
        ax[i].set_yscale("log")

# ax[0].set_ylabel(r"$y / c$")
if useDelta:
    ax[0].set_ylabel(r"$y / \delta$")
else:
    ax[0].set_ylabel(r"$y^+$")
ax[0].legend([r"$P_{\tilde{\nu}} / D_{\tilde{\nu}}$"], loc="upper right")

fig.legend(ncol=4, loc="lower center")
fig.tight_layout()
fig.subplots_adjust(bottom=0.2)
plt.savefig(join(save_dir, f"production_destruction_ratios_profiles_with_ratio_log_cl_{label}_scaled_with_{"delta" if useDelta else "yPlus"}_second.png"))
plt.show()

In [ ]:
# plot the production & destruction terms
fig, ax = plt.subplots(nrows= 1, ncols=4, figsize=(6, 3), sharey="row")
# ax2 = [a.twiny() for a in ax]

for c in range(len(cases)):
    for i in range(len(locations)):
        # compute approx. BL height
        if useDelta:
            sf, _ = compute_bl_thickness(all_u[c][i], all_z[c][i])
        else:
            sf = all_yPlus[c][i]

        # compute total destruction
        d = destruction[c][i] + compressibility[c][i]

        if i == 0:
            ax[i].plot(production[c][i], all_z[c][i] / sf  if useDelta else sf, zorder=10, color="black", marker="none", ls=ls[c], label=legend[c])
            ax[i].plot(d, all_z[c][i] / sf  if useDelta else sf, zorder=10, color="red", marker="none", ls=ls[c])
        else:
            ax[i].plot(production[c][i], all_z[c][i] / sf  if useDelta else sf, zorder=10, color="black",marker="none", ls=ls[c])
            ax[i].plot(d, all_z[c][i] / sf  if useDelta else sf, zorder=10, color="red",marker="none", ls=ls[c])

        ax[i].grid(visible=True, which="major", linestyle="-", alpha=0.35, color="black", axis="both")
        ax[i].minorticks_on()
        ax[i].tick_params(axis="both", which="minor", bottom=True)
        ax[i].grid(visible=True, which="minor", linestyle="--", alpha=0.25, color="black", axis="both")
        ax[i].set_title(fr"$ x / c = {float(locations[i]):.2f}$")
        # ax[i].set_ylim(0, 0.06)
        ax[i].set_ylim(0, 1)
        # ax[i].set_ylim(0, 2500)
        ax[i].set_xlim(0, 200)
        # ax[i].set_xscale("log")
        ax[i].set_yscale("log")

# ax[0].set_ylabel(r"$y / c$")
if useDelta:
    ax[0].set_ylabel(r"$y / \delta$")
else:
    ax[0].set_ylabel(r"$y^+$")
ax[0].legend([r"$P_{\tilde{\nu}}$", r"$D_{\tilde{\nu}}$"])

fig.legend(ncol=4, loc="lower center")
fig.tight_layout()
fig.subplots_adjust(bottom=0.2)
plt.savefig(join(save_dir, f"production_destruction_profiles_with_ratio_with_CC_log_cl_{label}_scaled_with_{"delta" if useDelta else "yPlus"}.png"))
plt.show()